# Test

In this section, we evaluate the four trained **cross-platform sentiment classification models** on our **cross-platform sentiment dataset**.

## Evaluation Metrics  
We will assess:  
1. **Overall model performance** across all platforms.  
2. **Platform-specific performance** for each model on:  
   - **GitHub**  
   - **Jira**  
   - **Mailbox**  

## Results  
The evaluation will print:  
- **Overall accuracy** of each model.  
- **Performance breakdown per platform** for each model.  

In [1]:
import os
import re
import string
import random
import warnings
import argparse
import numpy as np
import pandas as pd
import torch
import time
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from io import StringIO
from unicodedata import category
from bs4 import BeautifulSoup
from markdown import markdown
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score,classification_report
from torch.utils.data import DataLoader, RandomSampler, Dataset
from transformers import (
    BertTokenizer, BertForSequenceClassification, BertForMaskedLM,
    XLNetTokenizer, XLNetForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification, RobertaForMaskedLM,
    AlbertTokenizer, AlbertForSequenceClassification, AlbertForMaskedLM,
    get_scheduler
)

# Changed because AdamW Depreciated
from torch.optim import AdamW
from api.preprocessing import *
from api.train import *
from api.test import *

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()

In [ ]:
# Evaluates four pretrained models 
current_directory = os.getcwd()
# Load test dataset
test_df = pd.read_csv(f'{current_directory}/test_df.csv')

MODEL_NAMES = ['bert', 'xlnet', 'Roberta', 'albert']
model_results = {}

# Define platform mapping
platforms = {0: "GitHub", 1: "Jira", 2: "Mailbox"}

# Evaluate each model
for i, model_name in enumerate(MODEL_NAMES):
    model_path = f"{current_directory}/{model_name}_model"
    print(f"Evaluating {model_name} model...for overall platform")

    # Get overall accuracy
    overall_accuracy = test_model(test_df, model_path, model_select=i)

    # Evaluate accuracy per platform
    platform_accuracies = {}
    for platform_id, platform_name in platforms.items():
        test_df_platform = test_df[test_df["Platform"] == platform_id]
        if not test_df_platform.empty:
            print(f"Evaluating {model_name} model...for {platform_name} platform")
            accuracy = test_model(test_df_platform, model_path, model_select=i)
            platform_accuracies[platform_name] = accuracy
        else:
            platform_accuracies[platform_name] = "No data"





### Generalization Performance of the Model (Table 3.3)
In this section, we evaluate the **Bert-CP** model's **generalization performance** on the existing datasets:  
- **GitHub Golden Rule Dataset**  
- **Stack Overflow Dataset**  

We will also compare the performance of the **BERT model** trained on **GitHub Golden Rule** and **Stack Overflow** datasets, with a focus on **cross-platform performance**. This comparison aims to validate the **superiority** of our model.

## Evaluation Process  
- **Bert-CP Model Evaluation**: We test the **Bert-CP** model on the **GitHub Golden Rule** and **Stack Overflow** datasets.
- **Cross-Platform Comparison**: We compare the performance of models trained on **GitHub Golden Rule** and **Stack Overflow** datasets across multiple platforms using the **BERT model**.

## Goals  
- To assess the **generalization** of the **Bert-CP** model across different datasets.
- To highlight the **superiority** of our cross-platform model over dataset-specific models.

In [5]:
# Evaluate how bert trained on different datasets

# Load test datasets
test_gh = pd.read_csv(f'{current_directory}/test_gh.csv')
test_so = pd.read_csv(f'{current_directory}/test_so.csv')

# Define model paths
bert_model_path = f"{current_directory}/bert_model"
gh_bert_model_path = f"{current_directory}/GH_bert_model"
so_bert_model_path = f"{current_directory}/SO_bert_model"

# Store results
model_results = {}

# 1. Validate bert_model on test_gh and test_so
print("Evaluating bert_model on GitHub test dataset...")
bert_on_gh = test_model(test_gh, bert_model_path, model_select=0)

print("Evaluating bert_model on Stack Overflow test dataset...")
bert_on_so = test_model(test_so, bert_model_path, model_select=0)

model_results["bert_model"] = {
    "test_gh Accuracy": bert_on_gh,
    "test_so Accuracy": bert_on_so
}

# 2. Validate GH_bert_model on test_so
print("Evaluating GH_bert_model on Stack Overflow test dataset...")
gh_bert_on_so = test_model(test_so, gh_bert_model_path, model_select=0)

model_results["GH_bert_model"] = {
    "test_so Accuracy": gh_bert_on_so
}

# 3. Validate SO_bert_model on test_gh
print("Evaluating SO_bert_model on GitHub test dataset...")
so_bert_on_gh = test_model(test_gh, so_bert_model_path, model_select=0)

model_results["SO_bert_model"] = {
    "test_gh Accuracy": so_bert_on_gh
}


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Evaluating bert_model on GitHub test dataset...


/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 